In [2]:
getwd()
setwd("/liulab/galib/dlbcl_manuscript/")
library(tidyverse)
library(dplyr)
library(DoubletFinder)
library(rBCS)
library(Seurat)
library(harmony)
library(viridis)
library(RColorBrewer)
library(Polychrome)
PurpleAndYellow()
library(ComplexHeatmap)
library(devtools)
library(presto)
library(ggplot2)
library(ggpubr)
library(readxl)
source("./scripts/scplot.R")

[1] "/liulab/galib/dlbcl_manuscript/scripts"

Warning message:
“package ‘tidyverse’ was built under R version 4.1.3”
── Attaching packages ─────────────────────────────────────── tidyverse 1.3.1 ──

✔ ggplot2 3.3.6      ✔ purrr   0.3.4 
✔ tibble  3.1.8      ✔ dplyr   1.0.10
✔ tidyr   1.2.0      ✔ stringr 1.4.1 
✔ readr   2.1.2      ✔ forcats 0.5.1 

Warning message:
“package ‘tidyr’ was built under R version 4.1.2”
Warning message:
“package ‘readr’ was built under R version 4.1.2”
Warning message:
“package ‘forcats’ was built under R version 4.1.3”
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()

Warning message:
“package ‘DoubletFinder’ was built under R version 4.1.3”
Warning message:
“package ‘rBCS’ was built under R version 4.1.3”
Attaching SeuratObject

Attaching sp

Warning message:
“package ‘harmony’ was built under R version 4.1.3”
Loading required package: Rcpp

Warning message:
“package ‘Rcpp’ was built under R v

[1] "#FF00FF" "#F400F4" "#EA00EA" "#DF00DF" "#D500D5" "#CA00CA" "#BF00BF"
 [8] "#B500B5" "#AA00AA" "#9F009F" "#950095" "#8A008A" "#800080" "#750075"
[15] "#6A006A" "#600060" "#550055" "#4A004A" "#400040" "#350035" "#2B002B"
[22] "#200020" "#150015" "#0B000B" "#000000" "#000000" "#0B0B00" "#151500"
[29] "#202000" "#2B2B00" "#353500" "#404000" "#4A4A00" "#555500" "#606000"
[36] "#6A6A00" "#757500" "#808000" "#8A8A00" "#959500" "#9F9F00" "#AAAA00"
[43] "#B5B500" "#BFBF00" "#CACA00" "#D4D400" "#DFDF00" "#EAEA00" "#F4F400"
[50] "#FFFF00"

Warning message:
“package ‘ComplexHeatmap’ was built under R version 4.1.3”
Loading required package: grid

ComplexHeatmap version 2.10.0
Bioconductor page: http://bioconductor.org/packages/ComplexHeatmap/
Github page: https://github.com/jokergoo/ComplexHeatmap
Documentation: http://jokergoo.github.io/ComplexHeatmap-reference

If you use it in published research, please cite:
Gu, Z. Complex heatmaps reveal patterns and correlations in multidimensional 
  genomic data. Bioinformatics 2016.

The new InteractiveComplexHeatmap package can directly export static 
complex heatmaps into an interactive Shiny app with zero effort. Have a try!

This message can be suppressed by:
  suppressPackageStartupMessages(library(ComplexHeatmap))


Loading required package: usethis

Warning message:
“package ‘presto’ was built under R version 4.1.3”
Loading required package: data.table


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


The

In [2]:
preprocess_seurat<- function(obj){
  obj <- obj %>%
    NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
    FindVariableFeatures(selection.method = "vst", nfeatures = 2000) %>%
    ScaleData(vars.to.regress = "percent.mt") %>%
    RunPCA(npc = 50) %>%
    RunHarmony(group.by.vars = "pool_id") %>%
    RunUMAP(reduction = "harmony", dims = 1:50) %>%
    FindNeighbors(reduction = "harmony", dims = 1:50) %>%
    FindClusters(resolution = 1.5)
  return(obj)
}

In [ ]:
##############
### merged ###
##############
merged_final<- readRDS("./data/objects/merged_qc_doublet_rm.obj")

print("Start to recluster merged object after doublets removal...")

merged<- preprocess_seurat(merged_final)
saveRDS(merged, "./data/objects/merged_res1.5_obj.rds")

merged %>% dim()
# 32285 x 376307
print("merged space done!")

In [ ]:
##############
## CD3+ CD8- #
##############

print("Start to split cd3_pos_cd8_neg obj... ")

cd3_pos_cd8_neg_index_loose<- (merged[["RNA"]]@counts["Cd3d", ] != 0 |
                                 (merged[["RNA"]]@counts["Cd3g", ] != 0) |
                                 (merged[["RNA"]]@counts["Cd3e", ] != 0)) &
  merged[["RNA"]]@counts["Cd8a", ] == 0 &
  merged[["RNA"]]@counts["Cd8b1", ] == 0

cd3_pos_cd8_neg_index_strict<- (merged[["RNA"]]@counts["Cd3d", ] != 0 |
                                  (merged[["RNA"]]@counts["Cd3g", ] != 0) |
                                  (merged[["RNA"]]@counts["Cd3e", ] != 0)) &
  merged[["RNA"]]@counts["Cd8a", ] == 0 &
  merged[["RNA"]]@counts["Cd8b1", ] == 0 &
  merged[["RNA"]]@counts["Cd79b", ] == 0 &
  merged[["RNA"]]@counts["Cd19", ] == 0 &
  merged[["RNA"]]@counts["Pax5", ] == 0

cd3_pos_cd8_neg_index_loose  %>% table()
cd3_pos_cd8_neg_index_strict %>% table()

cd3_pos_cd8_neg<- merged[, cd3_pos_cd8_neg_index_strict]


print("Check markers == 0")
cd3_pos_cd8_neg[["RNA"]]@counts["Pax5",]  %>% table()
cd3_pos_cd8_neg[["RNA"]]@counts["Cd79b",]  %>% table()
cd3_pos_cd8_neg[["RNA"]]@counts["Cd19",]  %>% table()
cd3_pos_cd8_neg[["RNA"]]@counts["Cd8b1",]  %>% table()
cd3_pos_cd8_neg[["RNA"]]@counts["Cd8a",]  %>% table()

print("Check markers != 0")
cd3_pos_cd8_neg[["RNA"]]@counts["Cd4",]  %>% table()


cd3_pos_cd8_neg<- preprocess_seurat(cd3_pos_cd8_neg)
saveRDS(cd3_pos_cd8_neg, "./data/objects/cd3_pos_cd8_neg_res1.5_obj.rds")
ExportSeurat(cd3_pos_cd8_neg, "./data/objects/cd3_pos_cd8_neg_res1.5_obj.bcs", overwrite=TRUE)


cd3_pos_cd8_neg %>% dim()
# 32285 genes x 66275 cells
print("cd3_pos_cd8_neg space done!")

In [ ]:
##############
## CD3+ CD4- #
##############
print("Start to split out cd3_pos_cd4_neg obj... ")

cd3_pos_cd4_neg_index_loose<- (merged[["RNA"]]@counts["Cd3d", ] != 0 |
                                 (merged[["RNA"]]@counts["Cd3g", ] != 0) |
                                 (merged[["RNA"]]@counts["Cd3e", ] != 0)) &
  merged[["RNA"]]@counts["Cd4", ] == 0

cd3_pos_cd4_neg_index_strict<- (merged[["RNA"]]@counts["Cd3d", ] != 0 |
                                  (merged[["RNA"]]@counts["Cd3g", ] != 0) |
                                  (merged[["RNA"]]@counts["Cd3e", ] != 0)) &
  merged[["RNA"]]@counts["Cd4", ] == 0 &
  merged[["RNA"]]@counts["Cd79b", ] == 0 &
  merged[["RNA"]]@counts["Cd19", ] == 0 &
  merged[["RNA"]]@counts["Pax5", ] == 0

cd3_pos_cd4_neg <- merged[, cd3_pos_cd4_neg_index_strict]
cd3_pos_cd4_neg %>% dim()

print("Get shared cell barcodes of cd3_pos_cd4_neg and cd3_pos_cd8_neg space...")
common_bc<- intersect(colnames(cd3_pos_cd4_neg), colnames(cd3_pos_cd8_neg))

common_bc  %>% length()

print("Remove shared cell barcodes from cd3_pos_cd4_neg space...")

common_index<- !(colnames(cd3_pos_cd4_neg) %in% common_bc)
common_index  %>% table()
cd3_pos_cd4_neg<- cd3_pos_cd4_neg[, common_index]

print("Dim after removing shared cells...")

cd3_pos_cd4_neg %>% dim()

print("Check markers == 0")
cd3_pos_cd4_neg[["RNA"]]@counts["Pax5",]  %>% table()
cd3_pos_cd4_neg[["RNA"]]@counts["Cd79b",]  %>% table()
cd3_pos_cd4_neg[["RNA"]]@counts["Cd19",]  %>% table()
cd3_pos_cd4_neg[["RNA"]]@counts["Cd4",]  %>% table()

print("Check markers != 0")
cd3_pos_cd4_neg[["RNA"]]@counts["Cd3e",]  %>% table()
cd3_pos_cd4_neg[["RNA"]]@counts["Cd8b1",]  %>% table()
cd3_pos_cd4_neg[["RNA"]]@counts["Cd8a",]  %>% table()


cd3_pos_cd4_neg<- preprocess_seurat(cd3_pos_cd4_neg)
saveRDS(cd3_pos_cd4_neg, "./data/objects/cd3_pos_cd4_neg_res1.5_obj.rds")

ExportSeurat(cd3_pos_cd4_neg, "./data/objects/cd3_pos_cd4_neg_res1.5_obj.bcs", overwrite=TRUE)
cd3_pos_cd4_neg %>% dim()
# 32285 genes x 38842 cells
print("cd3_pos_cd4_neg space done! ")

In [ ]:
############
## B cell ##
############

print("Start to split B-cell space... ")

B_cell_index_loose<- (merged[["RNA"]]@counts["Cd79b", ] != 0 |
                        (merged[["RNA"]]@counts["Cd19", ] != 0) |
                        (merged[["RNA"]]@counts["Pax5", ] != 0))


T_cell_index <- (merged[["RNA"]]@counts["Cd4", ] != 0 |
                   (merged[["RNA"]]@counts["Cd8a", ] != 0) |
                   (merged[["RNA"]]@counts["Cd8b1", ] != 0) |
                   (merged[["RNA"]]@counts["Cd3e", ] != 0) |
                   (merged[["RNA"]]@counts["Cd3d", ] != 0) |
                   (merged[["RNA"]]@counts["Cd3g", ] != 0))


T_cell_index  %>% table()
table(!T_cell_index & B_cell_index_loose)
B_cell_index_strict <- (!T_cell_index) & B_cell_index_loose

B_cell_index_loose %>% table()
B_cell_index_strict %>% table()

B_obj<- merged[,B_cell_index_strict]


print("Check markers ==0")
B_obj[["RNA"]]@counts["Cd4", ]  %>% table()
B_obj[["RNA"]]@counts["Cd8a",]  %>% table()
B_obj[["RNA"]]@counts["Cd8b1",]  %>% table()
B_obj[["RNA"]]@counts["Cd3e",]  %>% table()
B_obj[["RNA"]]@counts["Cd3d",]  %>% table()
B_obj[["RNA"]]@counts["Cd3g",]  %>% table()

print("Check markers !=0")
B_obj[["RNA"]]@counts["Pax5", ]  %>% table()
B_obj[["RNA"]]@counts["Cd19",]  %>% table()
B_obj[["RNA"]]@counts["Cd79b",]  %>% table()


B_obj<- preprocess_seurat(B_obj)
saveRDS(B_obj, "./data/objects/B_cell_res1.5_obj.rds")
ExportSeurat(B_obj, "./data/objects/B_cell_res1.5_obj.bcs", overwrite=TRUE)

B_obj %>% dim()
# 32285 genes x 187259 cells
print("B-cell space done! ")

In [4]:
##################
## Feature Plot ##
##################

B_cell_features <- c("Pax5", "Cd19", "Cd79b")
B_Ig_features <- c("Ighm", "Ighd", "Igkc")

cd3_features <-  c("Cd3e", "Cd3g", "Cd3d")
cd8_features <- c("Cd8b1", "Cd8a")
cd4_features <- c("Cd4")

mono_macro_features <- c("Cd68", "Fcgr3", "Cd14")


merged_featureplot <- FeaturePlot(merged,
                                  features = c(B_cell_features,
                                               B_Ig_features,
                                               cd3_features,
                                               cd8_features,
                                               cd4_features,
                                               mono_macro_features),
                                  keep.scale = NULL, ncol =4)

cd3_pos_cd4_neg_featureplot <- FeaturePlot(cd3_pos_cd4_neg,
                                           features = c(B_cell_features,
                                                        B_Ig_features,
                                                        cd3_features,
                                                        cd8_features,
                                                        cd4_features,
                                                        mono_macro_features),
                                           keep.scale = NULL, ncol =4)

cd3_pos_cd8_neg_featureplot <- FeaturePlot(cd3_pos_cd8_neg,
                                           features = c(B_cell_features,
                                                        B_Ig_features,
                                                        cd3_features,
                                                        cd8_features,
                                                        cd4_features,
                                                        mono_macro_features),
                                           keep.scale = NULL, ncol =4)

B_cell_featureplot <- FeaturePlot(B_obj,
                                  features = c(B_cell_features,
                                               B_Ig_features,
                                               cd3_features,
                                               cd8_features,
                                               cd4_features,
                                               mono_macro_features),
                                  keep.scale = NULL, ncol =4)

ggsave("results/figures/1_merged_all_clusters_featureplot.pdf",merged_featureplot, width = 12, height = 10)
ggsave("results/figures/1_cd3_pos_cd4_neg_all_clusters_featureplot.pdf", cd3_pos_cd4_neg_featureplot, width = 12, height = 10)
ggsave("results/figures/1_cd3_pos_cd8_neg_all_clusters_featureplot.pdf", cd3_pos_cd8_neg_featureplot, width = 12, height = 10)
ggsave("results/figures/1_B_cell_all_clusters_featureplot.pdf", B_cell_featureplot, width = 12, height = 10)

### Umap on new data objects

In [5]:
DimPlot(cd3_pos_cd4_neg, reduction = "umap",
               label = TRUE, pt.size = 0.2 ) +
labs(title = "CD3+ CD4- space umap by clusters res=1.5", y = NULL, x = NULL) +
theme(text = element_text(size = 20))

ggsave("./results/figures/1_cd3_pos_cd4_neg_all_clusters_umap.pdf", width = 10, height = 8)

DimPlot(cd3_pos_cd8_neg, reduction = "umap",
               label = TRUE, pt.size = 0.2 ) +
labs(title = "CD3+ CD8- space umap by clusters res=1.5", y = NULL, x = NULL) +
theme(text = element_text(size = 20))

ggsave("./results/figures/1_cd3_pos_cd8_neg_all_clusters_umap.pdf", width = 10, height = 8)

DimPlot(B_cell, reduction = "umap",
               label = TRUE, pt.size = 0.2 ) +
labs(title = "B cell space umap by clusters res=1.5", y = NULL, x = NULL) +
theme(text = element_text(size = 20))

ggsave("./results/figures/1_B_cell_all_clusters_umap.pdf", width = 10, height = 8)

## Find marker in each cluster

In [5]:
# read seurat
cd3_pos_cd4_neg <- readRDS("./data/objects/cd3_pos_cd4_neg_res1.5_obj.rds")
cd3_pos_cd8_neg <- readRDS("./data/objects/cd3_pos_cd8_neg_res1.5_obj.rds")
B_cell <- readRDS("./data/objects/B_cell_res1.5_obj.rds")

In [ ]:
# CD3- CD4-
space = "cd3_pos_cd4_neg"
hp = plot_bi_clustered_heatmap(cd3_pos_cd4_neg, space = space, fsuffix = "all_clusters")
## Save to file
pdf(paste0("results/figures/1_", space, "_all_clusters_marker_gene_complex_heatmap.pdf"), width = 10, height = 30)
draw(hp$p1)    
dev.off()
# ##Plot top5 markers
pdf(paste0("results/figures/1_", space, "_all_clusters_top5_marker_gene_complex_heatmap.pdf"), width = 10, height = 8)
draw(hp$p2)    
dev.off()
# ##Plot top10 markers
pdf(paste0("results/figures/1_", space, "_all_clusters_top10_marker_gene_complex_heatmap.pdf"), width = 10, height = 10)
draw(hp$p3) 
dev.off()


# CD3- CD8-
space = "cd3_pos_cd8_neg"
hp = plot_bi_clustered_heatmap(cd3_pos_cd8_neg, space = space, fsuffix = "all_clusters")
## Save to file
pdf(paste0("results/figures/1_", space, "_all_clusters_marker_gene_complex_heatmap.pdf"), width = 10, height = 30)
draw(hp$p1)    
dev.off()
# ##Plot top5 markers
pdf(paste0("results/figures/1_", space, "_all_clusters_top5_markerfinal_gene_complex_heatmap.pdf"), width = 10, height = 8)
draw(hp$p2)
dev.off()
# ##Plot top10 markers
pdf(paste0("results/figures/1_", space, "_all_clusters_top10_marker_gene_complex_heatmap.pdf"), width = 10, height = 10)
draw(hp$p3) 
dev.off()

# B cell
space = "B_cell"
hp = plot_bi_clustered_heatmap(B_cell, space = space, fsuffix = "all_clusters")
## Save to file
pdf(paste0("results/figures/1_", space, "_all_clusters_marker_gene_complex_heatmap.pdf"), width = 10, height = 30)
draw(hp$p1)    
dev.off()
# ##Plot top5 markers
pdf(paste0("results/figures/1_", space, "_all_clusters_top5_marker_gene_complex_heatmap.pdf"), width = 10, height = 8)
draw(hp$p2)    
dev.off()
# ##Plot top10 markers
pdf(paste0("results/figures/1_", space, "_all_clusters_top10_marker_gene_complex_heatmap.pdf"), width = 10, height = 10)
draw(hp$p3)
dev.off()